In [85]:
# 1. HÜCRE: GEREKLİ KÜTÜPHANELER VE LOKAL MODEL KURULUMU
print("1. DSPy kütüphanesi kuruluyor...")
!pip install -q dspy-ai

print("2. Ollama sunucusu kuruluyor...")
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess
import time

print("3. Ollama arka planda başlatılıyor...")
# Ollama'yı arka planda sessizce çalıştırıyoruz
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5) # Sunucunun tam ayağa kalkması için 5 saniye bekleme

print("4. Meta Llama 3.2 modeli indiriliyor (Bu işlem 1-2 dakika sürebilir)...")
!ollama pull llama3.2
print("✅ Kurulum tamamlandı! Lokal model kullanıma hazır.")


1. DSPy kütüphanesi kuruluyor...
2. Ollama sunucusu kuruluyor...
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
3. Ollama arka planda başlatılıyor...
4. Meta Llama 3.2 modeli indiriliyor (Bu işlem 1-2 dakika sürebilir)...

✅ Kurulum tamamlandı! Lokal model kullanıma hazır.


In [126]:
# 2. HÜCRE: GELİŞMİŞ RAPORLAMA VE VERİ ÇIKARIMI
import dspy
import email
from email import policy
from email.parser import BytesParser

# --- A. DİL MODELİ YAPILANDIRMASI ---
lokal_model = dspy.LM('ollama/llama3.2', api_base='http://localhost:11434', max_tokens=2000)
dspy.settings.configure(lm=lokal_model)

# --- B. EML DOSYASI OKUMA FONKSİYONU ---
def parse_eml(file_path):
    with open(file_path, 'rb') as fp:
        msg = BytesParser(policy=policy.default).parse(fp)
    gonderici = msg.get('from', 'Bilinmiyor')
    konu = msg.get('subject', 'Bilinmiyor')
    tarih = msg.get('date', 'Bilinmiyor')
    mesaj_govdesi = ""
    if msg.is_multipart():
        for part in msg.walk():
            if part.get_content_type() == 'text/plain':
                payload = part.get_payload(decode=True)
                charset = part.get_content_charset() or 'utf-8'
                mesaj_govdesi = payload.decode(charset, errors='ignore')
                break
    else:
        payload = msg.get_payload(decode=True)
        charset = msg.get_content_charset() or 'utf-8'
        mesaj_govdesi = payload.decode(charset, errors='ignore')
    return gonderici, konu, tarih, mesaj_govdesi

# --- C. DSPy SIGNATURE (TAMAMEN ÖZGÜR) ---
class PhishingAnalyzer(dspy.Signature):
    """E-postayı siber güvenlik açısından analiz et."""

    gonderici = dspy.InputField()
    konu = dspy.InputField()
    mesaj_govdesi = dspy.InputField()

    risk_skoru = dspy.OutputField(desc="Risk puanı (1-10)")
    analiz_sonucu = dspy.OutputField(desc="Güvenlik analizi açıklaması")

# --- D. DSPy MODÜLÜ (TALİMATSIZ) ---
class PhishingModule(dspy.Module):
    def __init__(self):
        super().__init__()
        self.analyzer = dspy.Predict(PhishingAnalyzer)

    def forward(self, gonderici, konu, mesaj_govdesi):
        # Model tamamen kendi bilgisiyle hareket eder
        return self.analyzer(gonderici=gonderici, konu=konu, mesaj_govdesi=mesaj_govdesi)

In [128]:
# 3. HÜCRE: ANALİZİ ÇALIŞTIRMA
dosya_adi = "Süper Bahar Fırsatları idefix'te.eml"

try:
    gonderen, konu, tarih, govde = parse_eml(dosya_adi)
    ai_analist = PhishingModule()
    sonuc = ai_analist(gonderici=gonderen, konu=konu, mesaj_govdesi=govde)

    print("🛡️ SİBER GÜVENLİK ANALİZİ")
    print(f"👤 GÖNDEREN: {gonderen}")
    print(f"🚨 RİSK PUANI: {sonuc.risk_skoru}")
    print("-"*30)
    print(f"📝 ANALİZ: {sonuc.analiz_sonucu}")

except Exception as e:
    print(f"Hata: {e}")

🛡️ SİBER GÜVENLİK ANALİZİ
👤 GÖNDEREN: idefix <info@ed.idefix.com>
🚨 RİSK PUANI: 6
------------------------------
📝 ANALİZ: E-postanın content-type ve accept headersinin doğru being used being indicated. Additionally, e-posta body's inlined images' Content-Transfer-Encoding being set to base64. These are normal practices for email security and should not trigger any security concerns.
